
# Heston Delta--Vega Neural Hedging Extension

This notebook implements a report-ready multi-instrument neural hedging extension:

\[
\text{Heston stock dynamics} + \text{stock and liquid option as hedging instruments}.
\]

The goal is to move beyond the frictionless Black--Scholes setting, where the stock alone spans the single Brownian risk factor. Under Heston dynamics, the stock price is driven by stochastic variance, so the target option is exposed to both underlying-price risk and volatility/variance risk.

A second traded option is therefore interpreted primarily as a **vega hedging instrument**, rather than merely a gamma hedge.

The Heston dynamics are

\[
dS_t = rS_t\,dt + \sqrt{v_t}S_t\,dW_t^S,
\]

\[
dv_t = \kappa(\theta-v_t)\,dt + \xi\sqrt{v_t}\,dW_t^v,
\]

with

\[
dW_t^S dW_t^v = \rho\,dt.
\]

The stock-and-option terminal wealth is

\[
V_T
=
\pi
+
\sum_{n=0}^{N-1}\delta_n(S_{t_{n+1}}-S_{t_n})
+
\sum_{n=0}^{N-1}\eta_n(C^h_{t_{n+1}}-C^h_{t_n}),
\]

where \(C^h\) is a liquid hedging option and the neural network outputs

\[
(\delta_n,\eta_n).
\]

## Important modelling note

For computational tractability, the liquid hedging option is priced using a **Black--Scholes proxy with instantaneous volatility** \(\sqrt{v_t}\):

\[
C^h_t \approx C_{BS}(S_t,K_h,T_h-t,\sqrt{v_t}).
\]

This makes the extension practical and keeps hedge-instrument prices available along the simulated Heston paths. In the report, this should be described as a **liquid-option proxy** rather than exact Heston semi-closed-form pricing.


> **Revision note:** A technical-review revision block has been appended at the end of this notebook to add proxy-drift diagnostics, stock-only tanh control, fair-premium evaluation, and revised report outputs.

> **Second revision note:** The interpretation has been corrected so that raw Mean HE is no longer treated as a primary proxy-drift detector. A superseded-results banner and optional multi-seed headline robustness cell have also been added.

In [ ]:

# ============================================================
# Configuration
# ============================================================

RUN_FULL = False  # Set True for report-quality runs

# Target option and grid
S0 = 1.0
K_TARGET = 0.9
T_TARGET = 0.5
R = 0.0
N_STEPS = 125

# Heston parameters
V0 = 0.16          # initial variance, sqrt(V0)=0.40
KAPPA = 2.0       # mean reversion speed
THETA = 0.16      # long-run variance
XI = 0.60         # vol-of-vol
RHO = -0.70       # spot/vol correlation

# Liquid hedging option proxy.
# It should mature after the target option so it remains tradeable over [0,T_TARGET].
K_HEDGE = 1.0
T_HEDGE = 1.0

# Neural architecture
HIDDEN_WIDTH = 64
HIDDEN_DEPTH = 3
DELTA_SCALE = 5.0      # stock-position output range: approximately [-5,5]
ETA_SCALE = 5.0        # option-position output range: approximately [-5,5]

# Greek proxy clipping for stability
DELTA_CLIP = 5.0
ETA_CLIP = 5.0

# Training sizes
if RUN_FULL:
    N_TRAIN = 100_000
    N_VAL = 25_000
    N_TEST = 100_000
    MAX_EPOCHS = 150
    BATCH_SIZE = 4096
    PATIENCE = 30
else:
    N_TRAIN = 20_000
    N_VAL = 10_000
    N_TEST = 50_000
    MAX_EPOCHS = 60
    BATCH_SIZE = 2048
    PATIENCE = 15

LEARNING_RATE = 1e-3
SEED = 123

OUTPUT_DIR = "heston_delta_vega_outputs"
RESULTS_CSV = "heston_delta_vega_results.csv"


In [ ]:

# ============================================================
# Imports and device
# ============================================================

import math
import random
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

torch.manual_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)


In [ ]:

# ============================================================
# Black--Scholes proxy pricing utilities
# ============================================================

def norm_cdf(x):
    return 0.5 * (1.0 + torch.erf(x / math.sqrt(2.0)))


def norm_pdf(x):
    return torch.exp(-0.5 * x * x) / math.sqrt(2.0 * math.pi)


def bs_call_price(S, K, tau, vol, r=R):
    """
    Vectorised Black--Scholes call price.
    S, tau and vol can be tensors with broadcast-compatible shapes.
    """
    tau = torch.as_tensor(tau, dtype=S.dtype, device=S.device)
    vol = torch.as_tensor(vol, dtype=S.dtype, device=S.device)

    tau_safe = torch.clamp(tau, min=1e-10)
    vol_safe = torch.clamp(vol, min=1e-6)
    sqrt_tau = torch.sqrt(tau_safe)

    d1 = (torch.log(torch.clamp(S, min=1e-12) / K) + (r + 0.5 * vol_safe**2) * tau_safe) / (vol_safe * sqrt_tau)
    d2 = d1 - vol_safe * sqrt_tau

    disc_K = K * torch.exp(torch.tensor(-r, dtype=S.dtype, device=S.device) * tau_safe)
    price = S * norm_cdf(d1) - disc_K * norm_cdf(d2)
    payoff = torch.clamp(S - K, min=0.0)
    return torch.where(tau > 1e-8, price, payoff)


def bs_delta(S, K, tau, vol, r=R):
    tau = torch.as_tensor(tau, dtype=S.dtype, device=S.device)
    vol = torch.as_tensor(vol, dtype=S.dtype, device=S.device)

    tau_safe = torch.clamp(tau, min=1e-10)
    vol_safe = torch.clamp(vol, min=1e-6)
    sqrt_tau = torch.sqrt(tau_safe)

    d1 = (torch.log(torch.clamp(S, min=1e-12) / K) + (r + 0.5 * vol_safe**2) * tau_safe) / (vol_safe * sqrt_tau)
    delta = norm_cdf(d1)
    terminal_delta = (S > K).to(S.dtype)
    return torch.where(tau > 1e-8, delta, terminal_delta)


def bs_vega(S, K, tau, vol, r=R):
    """
    Black--Scholes vega with respect to volatility sigma.
    """
    tau = torch.as_tensor(tau, dtype=S.dtype, device=S.device)
    vol = torch.as_tensor(vol, dtype=S.dtype, device=S.device)

    tau_safe = torch.clamp(tau, min=1e-10)
    vol_safe = torch.clamp(vol, min=1e-6)
    sqrt_tau = torch.sqrt(tau_safe)

    d1 = (torch.log(torch.clamp(S, min=1e-12) / K) + (r + 0.5 * vol_safe**2) * tau_safe) / (vol_safe * sqrt_tau)
    vega = S * norm_pdf(d1) * sqrt_tau
    return torch.where(tau > 1e-8, vega, torch.zeros_like(S))


def scalar_bs_proxy_price(S0, K, T, vol0):
    S = torch.tensor(float(S0), device=device)
    tau = torch.tensor(float(T), device=device)
    vol = torch.tensor(float(vol0), device=device)
    return float(bs_call_price(S, K, tau, vol).detach().cpu())


In [ ]:

# ============================================================
# Heston path simulation
# ============================================================

def simulate_heston_paths(n_paths, seed, device=device):
    """
    Simulate risk-neutral Heston paths using a full-truncation Euler scheme.

    Returns:
        S: stock paths, shape [n_paths, N_STEPS+1]
        v: variance paths, shape [n_paths, N_STEPS+1]
        C_h: liquid hedging-option proxy price paths, shape [n_paths, N_STEPS+1]
        times: grid times, shape [N_STEPS+1]
    """
    torch.manual_seed(seed)

    dt = T_TARGET / N_STEPS
    sqrt_dt = math.sqrt(dt)

    S = torch.empty(n_paths, N_STEPS + 1, device=device)
    v = torch.empty(n_paths, N_STEPS + 1, device=device)

    S[:, 0] = S0
    v[:, 0] = V0

    for n in range(N_STEPS):
        z1 = torch.randn(n_paths, device=device)
        z2 = torch.randn(n_paths, device=device)

        z_s = z1
        z_v = RHO * z1 + math.sqrt(max(1.0 - RHO**2, 0.0)) * z2

        v_pos = torch.clamp(v[:, n], min=0.0)

        # Log-Euler for stock using current variance.
        S[:, n+1] = S[:, n] * torch.exp((R - 0.5 * v_pos) * dt + torch.sqrt(v_pos) * sqrt_dt * z_s)

        # Full-truncation Euler for variance.
        v_next = v[:, n] + KAPPA * (THETA - v_pos) * dt + XI * torch.sqrt(v_pos) * sqrt_dt * z_v
        v[:, n+1] = torch.clamp(v_next, min=0.0)

    times = torch.linspace(0.0, T_TARGET, N_STEPS + 1, device=device)

    # Liquid option proxy price under BS with instantaneous vol sqrt(v_t).
    tau_h = (T_HEDGE - times).view(1, -1).expand_as(S)
    inst_vol = torch.sqrt(torch.clamp(v, min=1e-10))
    C_h = bs_call_price(S, K_HEDGE, tau_h, inst_vol)

    return S, v, C_h, times


S_train, v_train, C_train, times = simulate_heston_paths(N_TRAIN, SEED + 1)
S_val, v_val, C_val, _ = simulate_heston_paths(N_VAL, SEED + 2)
S_test, v_test, C_test, _ = simulate_heston_paths(N_TEST, SEED + 3)

payoff_train = torch.clamp(S_train[:, -1] - K_TARGET, min=0.0)
payoff_val = torch.clamp(S_val[:, -1] - K_TARGET, min=0.0)
payoff_test = torch.clamp(S_test[:, -1] - K_TARGET, min=0.0)

# Heston target premium estimated by Monte Carlo on training set.
# The neural models also learn their own premium, initialized at this estimate.
target_premium_mc = float(torch.mean(payoff_train).detach().cpu())
hedge_option_proxy_0 = scalar_bs_proxy_price(S0, K_HEDGE, T_HEDGE, math.sqrt(V0))

print(f"MC target call premium estimate under Heston: {target_premium_mc:.6f}")
print(f"Initial liquid-option proxy price: {hedge_option_proxy_0:.6f}")
print("Train S shape:", tuple(S_train.shape), "Train v shape:", tuple(v_train.shape))
print("Mean terminal stock:", float(S_train[:, -1].mean().detach().cpu()))
print("Mean terminal variance:", float(v_train[:, -1].mean().detach().cpu()))



## Heston parameter table and sample-path diagnostics

This cell records the stochastic-volatility parameters used in the experiment and generates two report-friendly diagnostic figures:

1. sample stock-price paths under Heston dynamics;
2. sample variance paths under Heston dynamics.

These figures are useful in the report because they make clear that this extension no longer uses constant-volatility Black--Scholes stock dynamics.


In [ ]:

# ============================================================
# Heston parameter table and sample-path diagnostics
# ============================================================

parameter_rows = [
    {"Parameter": r"$S_0$", "Value": S0, "Description": "Initial stock price"},
    {"Parameter": r"$K$", "Value": K_TARGET, "Description": "Target option strike"},
    {"Parameter": r"$T$", "Value": T_TARGET, "Description": "Target option maturity"},
    {"Parameter": r"$r$", "Value": R, "Description": "Risk-free rate"},
    {"Parameter": r"$N$", "Value": N_STEPS, "Description": "Number of rebalancing dates"},
    {"Parameter": r"$v_0$", "Value": V0, "Description": "Initial variance"},
    {"Parameter": r"$\kappa$", "Value": KAPPA, "Description": "Variance mean-reversion speed"},
    {"Parameter": r"$\theta$", "Value": THETA, "Description": "Long-run variance"},
    {"Parameter": r"$\xi$", "Value": XI, "Description": "Volatility of variance"},
    {"Parameter": r"$\rho$", "Value": RHO, "Description": "Spot/variance Brownian correlation"},
    {"Parameter": r"$K_h$", "Value": K_HEDGE, "Description": "Liquid hedging-option strike"},
    {"Parameter": r"$T_h$", "Value": T_HEDGE, "Description": "Liquid hedging-option maturity"},
]

heston_parameter_table = pd.DataFrame(parameter_rows)
heston_parameter_table.to_csv(Path(OUTPUT_DIR) / "heston_parameter_table.csv", index=False)
display(heston_parameter_table)

print("LaTeX parameter table:")
print(heston_parameter_table.to_latex(index=False, escape=False, float_format=lambda x: f"{x:.4f}"))

# Sample-path figures
n_plot = min(20, S_train.shape[0])
idx = torch.linspace(0, S_train.shape[0] - 1, n_plot).long()
times_np = times.detach().cpu().numpy()

plt.figure(figsize=(8, 5))
for i in idx:
    plt.plot(times_np, S_train[i].detach().cpu().numpy(), linewidth=1.0, alpha=0.8)
plt.xlabel("Time")
plt.ylabel("Stock price")
plt.title("Sample Heston stock-price paths")
plt.tight_layout()
plt.savefig(Path(OUTPUT_DIR) / "heston_sample_stock_paths.png", dpi=200)
plt.show()

plt.figure(figsize=(8, 5))
for i in idx:
    plt.plot(times_np, v_train[i].detach().cpu().numpy(), linewidth=1.0, alpha=0.8)
plt.xlabel("Time")
plt.ylabel("Variance")
plt.title("Sample Heston variance paths")
plt.tight_layout()
plt.savefig(Path(OUTPUT_DIR) / "heston_sample_variance_paths.png", dpi=200)
plt.show()


In [ ]:

# ============================================================
# Feature construction
# ============================================================

def stock_only_features(S, v, times):
    """
    Stock-only Heston hedge features:
        log(S/K_target),
        tau_target/T_target,
        v/theta
    Shape: [B, N_STEPS, 3]
    """
    B = S.shape[0]
    S_n = S[:, :-1]
    v_n = v[:, :-1]
    tau_target = (T_TARGET - times[:-1]).view(1, -1).expand(B, -1)

    return torch.stack([
        torch.log(torch.clamp(S_n, min=1e-12) / K_TARGET),
        tau_target / T_TARGET,
        v_n / THETA,
    ], dim=-1)


def stock_option_features(S, v, C_h, times):
    """
    Stock + liquid-option Heston hedge features:
        log(S/K_target),
        tau_target/T_target,
        v/theta,
        log(S/K_hedge),
        tau_hedge/T_hedge,
        C_h/S

    Shape: [B, N_STEPS, 6]
    """
    B = S.shape[0]
    S_n = S[:, :-1]
    v_n = v[:, :-1]
    C_n = C_h[:, :-1]

    tau_target = (T_TARGET - times[:-1]).view(1, -1).expand(B, -1)
    tau_hedge = (T_HEDGE - times[:-1]).view(1, -1).expand(B, -1)

    return torch.stack([
        torch.log(torch.clamp(S_n, min=1e-12) / K_TARGET),
        tau_target / T_TARGET,
        v_n / THETA,
        torch.log(torch.clamp(S_n, min=1e-12) / K_HEDGE),
        tau_hedge / T_HEDGE,
        C_n / torch.clamp(S_n, min=1e-12),
    ], dim=-1)


In [ ]:

# ============================================================
# Neural hedge models
# ============================================================

class StockOnlyHestonNN(nn.Module):
    """
    Stock-only neural hedge under Heston dynamics.
    Output is bounded to [0,1] as a natural call-delta-like position.
    """
    def __init__(self, hidden_width=64, hidden_depth=3):
        super().__init__()
        layers = []
        in_dim = 3
        for _ in range(hidden_depth):
            layers.append(nn.Linear(in_dim, hidden_width))
            layers.append(nn.Tanh())
            in_dim = hidden_width
        layers.append(nn.Linear(hidden_width, 1))
        self.net = nn.Sequential(*layers)
        self.premium = nn.Parameter(torch.tensor(target_premium_mc, dtype=torch.float32))

    def forward(self, x):
        B, N, D = x.shape
        raw = self.net(x.reshape(B * N, D)).reshape(B, N)
        delta = torch.sigmoid(raw)
        return delta


class StockOptionHestonNN(nn.Module):
    """
    Stock + liquid-option neural hedge under Heston dynamics.
    Outputs:
        delta_n: stock position
        eta_n: liquid-option position

    Tanh-scaled outputs are used because multi-instrument hedge positions
    need not lie in [0,1].
    """
    def __init__(self, hidden_width=64, hidden_depth=3, delta_scale=5.0, eta_scale=5.0):
        super().__init__()
        self.delta_scale = delta_scale
        self.eta_scale = eta_scale

        layers = []
        in_dim = 6
        for _ in range(hidden_depth):
            layers.append(nn.Linear(in_dim, hidden_width))
            layers.append(nn.Tanh())
            in_dim = hidden_width
        layers.append(nn.Linear(hidden_width, 2))
        self.net = nn.Sequential(*layers)
        self.premium = nn.Parameter(torch.tensor(target_premium_mc, dtype=torch.float32))

    def forward(self, x):
        B, N, D = x.shape
        raw = self.net(x.reshape(B * N, D)).reshape(B, N, 2)
        delta = self.delta_scale * torch.tanh(raw[..., 0])
        eta = self.eta_scale * torch.tanh(raw[..., 1])
        return delta, eta


In [ ]:

# ============================================================
# Hedge error functions
# ============================================================

def hedge_error_stock_only(model, S, v, C_h, payoff, times):
    x = stock_only_features(S, v, times)
    delta = model(x)
    eta = torch.zeros_like(delta)

    dS = S[:, 1:] - S[:, :-1]
    gains = torch.sum(delta * dS, dim=1)
    he = model.premium + gains - payoff

    positions = {"delta": delta, "eta": eta}
    return he, positions


def hedge_error_stock_option(model, S, v, C_h, payoff, times):
    x = stock_option_features(S, v, C_h, times)
    delta, eta = model(x)

    dS = S[:, 1:] - S[:, :-1]
    dC = C_h[:, 1:] - C_h[:, :-1]

    gains = torch.sum(delta * dS + eta * dC, dim=1)
    he = model.premium + gains - payoff

    positions = {"delta": delta, "eta": eta}
    return he, positions


def position_turnover(pos):
    prev = torch.cat([torch.zeros(pos.shape[0], 1, device=pos.device), pos[:, :-1]], dim=1)
    return torch.sum(torch.abs(pos - prev), dim=1)


In [ ]:

# ============================================================
# Training
# ============================================================

def train_model(model, kind, S_train, v_train, C_train, y_train, S_val, v_val, C_val, y_val, times):
    """
    kind: 'stock' or 'stock_option'
    """
    model = model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)

    best_val = float("inf")
    best_state = None
    bad_epochs = 0
    history = []

    n = S_train.shape[0]

    for epoch in range(1, MAX_EPOCHS + 1):
        model.train()
        perm = torch.randperm(n, device=device)
        batch_losses = []

        for start in range(0, n, BATCH_SIZE):
            idx = perm[start:start + BATCH_SIZE]
            optimizer.zero_grad()

            if kind == "stock":
                he, _ = hedge_error_stock_only(model, S_train[idx], v_train[idx], C_train[idx], y_train[idx], times)
            else:
                he, _ = hedge_error_stock_option(model, S_train[idx], v_train[idx], C_train[idx], y_train[idx], times)

            loss = torch.mean(he ** 2)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
            optimizer.step()
            batch_losses.append(float(loss.detach().cpu()))

        model.eval()
        with torch.no_grad():
            if kind == "stock":
                he_val, _ = hedge_error_stock_only(model, S_val, v_val, C_val, y_val, times)
            else:
                he_val, _ = hedge_error_stock_option(model, S_val, v_val, C_val, y_val, times)
            val_loss = float(torch.mean(he_val ** 2).detach().cpu())

        train_loss = float(np.mean(batch_losses))
        premium = float(model.premium.detach().cpu())
        history.append({"epoch": epoch, "train_loss": train_loss, "val_loss": val_loss, "premium": premium})

        if epoch == 1 or epoch % 10 == 0:
            print(f"{kind:12s} epoch {epoch:4d} | train {train_loss:.8e} | val {val_loss:.8e} | premium {premium:.6f}")

        if val_loss < best_val - 1e-10:
            best_val = val_loss
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            bad_epochs = 0
        else:
            bad_epochs += 1

        if bad_epochs >= PATIENCE:
            print(f"Early stopping {kind} at epoch {epoch}; best val {best_val:.8e}")
            break

    if best_state is not None:
        model.load_state_dict(best_state)

    return model, pd.DataFrame(history)


In [ ]:

# ============================================================
# Proxy analytic benchmark strategies
# ============================================================

@torch.no_grad()
def proxy_stock_delta_positions(S, v, times):
    """
    Black--Scholes proxy stock delta using instantaneous volatility sqrt(v_t).
    """
    B = S.shape[0]
    S_n = S[:, :-1]
    vol_n = torch.sqrt(torch.clamp(v[:, :-1], min=1e-10))
    tau_target = (T_TARGET - times[:-1]).view(1, -1).expand(B, -1)

    delta = bs_delta(S_n, K_TARGET, tau_target, vol_n)
    eta = torch.zeros_like(delta)
    return delta, eta


@torch.no_grad()
def proxy_delta_vega_positions(S, v, times):
    """
    Proxy delta-vega benchmark using BS Greeks with instantaneous volatility sqrt(v_t).

    Match vega using the liquid option:
        eta = Vega_target / Vega_hedge

    Then match delta:
        delta = Delta_target - eta * Delta_hedge

    This is only a proxy benchmark. It is not an exact Heston hedge.
    Positions are clipped for stability when hedge-option vega is small.
    """
    B = S.shape[0]
    S_n = S[:, :-1]
    vol_n = torch.sqrt(torch.clamp(v[:, :-1], min=1e-10))

    tau_target = (T_TARGET - times[:-1]).view(1, -1).expand(B, -1)
    tau_hedge = (T_HEDGE - times[:-1]).view(1, -1).expand(B, -1)

    delta_target = bs_delta(S_n, K_TARGET, tau_target, vol_n)
    vega_target = bs_vega(S_n, K_TARGET, tau_target, vol_n)

    delta_hedge = bs_delta(S_n, K_HEDGE, tau_hedge, vol_n)
    vega_hedge = bs_vega(S_n, K_HEDGE, tau_hedge, vol_n)

    eta = vega_target / torch.clamp(vega_hedge, min=1e-6)
    eta = torch.clamp(eta, min=-ETA_CLIP, max=ETA_CLIP)

    delta = delta_target - eta * delta_hedge
    delta = torch.clamp(delta, min=-DELTA_CLIP, max=DELTA_CLIP)

    return delta, eta


@torch.no_grad()
def evaluate_given_positions(name, premium, delta, eta, S, C_h, payoff):
    dS = S[:, 1:] - S[:, :-1]
    dC = C_h[:, 1:] - C_h[:, :-1]
    he = premium + torch.sum(delta * dS + eta * dC, dim=1) - payoff
    return summarize_strategy(name, premium, he, {"delta": delta, "eta": eta})


def summarize_strategy(name, premium, he, positions):
    he_np = he.detach().cpu().numpy()
    loss = -he_np

    delta = positions["delta"]
    eta = positions["eta"]

    stock_turn = position_turnover(delta).detach().cpu().numpy()
    option_turn = position_turnover(eta).detach().cpu().numpy()

    def cvar_upper(x, alpha):
        q = np.quantile(x, alpha)
        return x[x >= q].mean()

    return {
        "Strategy": name,
        "Premium": float(premium),
        "RMSE": float(np.sqrt(np.mean(he_np ** 2))),
        "Mean HE": float(np.mean(he_np)),
        "Std HE": float(np.std(he_np)),
        "HE q01": float(np.quantile(he_np, 0.01)),
        "HE q05": float(np.quantile(he_np, 0.05)),
        "Loss VaR95": float(np.quantile(loss, 0.95)),
        "Loss CVaR95": float(cvar_upper(loss, 0.95)),
        "Loss VaR99": float(np.quantile(loss, 0.99)),
        "Loss CVaR99": float(cvar_upper(loss, 0.99)),
        "Mean abs delta": float(torch.mean(torch.abs(delta)).detach().cpu()),
        "Mean abs eta": float(torch.mean(torch.abs(eta)).detach().cpu()),
        "Mean stock turnover": float(np.mean(stock_turn)),
        "Mean option turnover": float(np.mean(option_turn)),
        "Median stock turnover": float(np.median(stock_turn)),
        "Median option turnover": float(np.median(option_turn)),
    }


In [ ]:

# ============================================================
# Train models
# ============================================================

stock_model = StockOnlyHestonNN(HIDDEN_WIDTH, HIDDEN_DEPTH)
stock_option_model = StockOptionHestonNN(HIDDEN_WIDTH, HIDDEN_DEPTH, DELTA_SCALE, ETA_SCALE)

print("Training stock-only Heston neural hedge...")
stock_model, hist_stock = train_model(
    stock_model,
    "stock",
    S_train, v_train, C_train, payoff_train,
    S_val, v_val, C_val, payoff_val,
    times,
)

print("\nTraining stock + option Heston neural hedge...")
stock_option_model, hist_stock_option = train_model(
    stock_option_model,
    "stock_option",
    S_train, v_train, C_train, payoff_train,
    S_val, v_val, C_val, payoff_val,
    times,
)

hist_stock.to_csv(Path(OUTPUT_DIR) / "stock_only_heston_training_history.csv", index=False)
hist_stock_option.to_csv(Path(OUTPUT_DIR) / "stock_option_heston_training_history.csv", index=False)



## Superseded fixed-premium results below

The next original evaluation/table/plot cells are kept for reproducibility, but their fixed-premium results should **not** be used as the headline dissertation results.

The canonical headline results are produced later in the technical revision block using symmetric fair-premium evaluation:

```text
heston_revised_results_fair_premium.csv
```

Use the original cells only as historical/prototype diagnostics.


In [ ]:

# ============================================================
# Evaluate on common test set
# ============================================================

results = []

with torch.no_grad():
    zero = torch.zeros(N_TEST, N_STEPS, device=device)

    # No hedge
    results.append(evaluate_given_positions(
        "No hedge",
        target_premium_mc,
        zero,
        zero,
        S_test,
        C_test,
        payoff_test,
    ))

    # Proxy stock delta
    proxy_delta, proxy_eta_zero = proxy_stock_delta_positions(S_test, v_test, times)
    results.append(evaluate_given_positions(
        "BS proxy stock delta",
        target_premium_mc,
        proxy_delta,
        proxy_eta_zero,
        S_test,
        C_test,
        payoff_test,
    ))

    # Proxy delta-vega hedge
    dv_delta, dv_eta = proxy_delta_vega_positions(S_test, v_test, times)
    results.append(evaluate_given_positions(
        f"BS proxy delta-vega hedge (eta clipped {ETA_CLIP:g})",
        target_premium_mc,
        dv_delta,
        dv_eta,
        S_test,
        C_test,
        payoff_test,
    ))

    # Stock-only neural hedge
    he_stock, pos_stock = hedge_error_stock_only(stock_model, S_test, v_test, C_test, payoff_test, times)
    results.append(summarize_strategy(
        "Stock-only NN under Heston",
        float(stock_model.premium.detach().cpu()),
        he_stock,
        pos_stock,
    ))

    # Stock + option neural hedge
    he_stock_option, pos_stock_option = hedge_error_stock_option(stock_option_model, S_test, v_test, C_test, payoff_test, times)
    results.append(summarize_strategy(
        "Stock + option NN under Heston",
        float(stock_option_model.premium.detach().cpu()),
        he_stock_option,
        pos_stock_option,
    ))

results_df = pd.DataFrame(results)
results_df.to_csv(RESULTS_CSV, index=False)
results_df.to_csv(Path(OUTPUT_DIR) / RESULTS_CSV, index=False)

pd.set_option("display.max_columns", 40)
display(results_df)
print(f"Saved results to {RESULTS_CSV} and {Path(OUTPUT_DIR) / RESULTS_CSV}")


In [ ]:

# ============================================================
# Improvement summary
# ============================================================

def pct_improvement(old, new):
    return 100.0 * (old - new) / old

stock_nn = results_df.loc[results_df["Strategy"] == "Stock-only NN under Heston"].iloc[0]
stock_option_nn = results_df.loc[results_df["Strategy"] == "Stock + option NN under Heston"].iloc[0]
proxy_stock = results_df.loc[results_df["Strategy"] == "BS proxy stock delta"].iloc[0]
proxy_dv = results_df[results_df["Strategy"].str.startswith("BS proxy delta-vega")].iloc[0]

summary = pd.DataFrame([
    {
        "Comparison": "Stock + option NN vs stock-only NN",
        "RMSE improvement (%)": pct_improvement(stock_nn["RMSE"], stock_option_nn["RMSE"]),
        "Loss CVaR95 improvement (%)": pct_improvement(stock_nn["Loss CVaR95"], stock_option_nn["Loss CVaR95"]),
    },
    {
        "Comparison": "Stock + option NN vs BS proxy stock delta",
        "RMSE improvement (%)": pct_improvement(proxy_stock["RMSE"], stock_option_nn["RMSE"]),
        "Loss CVaR95 improvement (%)": pct_improvement(proxy_stock["Loss CVaR95"], stock_option_nn["Loss CVaR95"]),
    },
    {
        "Comparison": "Stock + option NN vs BS proxy delta-vega",
        "RMSE improvement (%)": pct_improvement(proxy_dv["RMSE"], stock_option_nn["RMSE"]),
        "Loss CVaR95 improvement (%)": pct_improvement(proxy_dv["Loss CVaR95"], stock_option_nn["Loss CVaR95"]),
    },
])
display(summary)
summary.to_csv(Path(OUTPUT_DIR) / "heston_delta_vega_improvement_summary.csv", index=False)


In [ ]:

# ============================================================
# Plots
# ============================================================

# RMSE bar plot
plot_df = results_df.sort_values("RMSE")
plt.figure(figsize=(9, 5))
plt.bar(plot_df["Strategy"], plot_df["RMSE"])
plt.xticks(rotation=30, ha="right")
plt.ylabel("RMSE")
plt.title("Heston stock-and-option hedging: RMSE comparison")
plt.tight_layout()
plt.savefig(Path(OUTPUT_DIR) / "heston_delta_vega_rmse_bar.png", dpi=200)
plt.show()

# Loss CVaR95 bar plot
plot_df = results_df.sort_values("Loss CVaR95")
plt.figure(figsize=(9, 5))
plt.bar(plot_df["Strategy"], plot_df["Loss CVaR95"])
plt.xticks(rotation=30, ha="right")
plt.ylabel("Loss CVaR95")
plt.title("Heston stock-and-option hedging: seller tail-risk comparison")
plt.tight_layout()
plt.savefig(Path(OUTPUT_DIR) / "heston_delta_vega_cvar95_bar.png", dpi=200)
plt.show()

# Hedge-error distributions for key strategies
with torch.no_grad():
    he_proxy_stock = target_premium_mc + torch.sum(proxy_delta * (S_test[:, 1:] - S_test[:, :-1]), dim=1) - payoff_test
    he_proxy_dv = target_premium_mc + torch.sum(
        dv_delta * (S_test[:, 1:] - S_test[:, :-1]) + dv_eta * (C_test[:, 1:] - C_test[:, :-1]),
        dim=1
    ) - payoff_test
    he_stock, _ = hedge_error_stock_only(stock_model, S_test, v_test, C_test, payoff_test, times)
    he_stock_option, _ = hedge_error_stock_option(stock_option_model, S_test, v_test, C_test, payoff_test, times)

he_dict = {
    "BS proxy stock delta": he_proxy_stock.detach().cpu().numpy(),
    "BS proxy delta-vega": he_proxy_dv.detach().cpu().numpy(),
    "Stock-only NN": he_stock.detach().cpu().numpy(),
    "Stock + option NN": he_stock_option.detach().cpu().numpy(),
}

all_he = np.concatenate(list(he_dict.values()))
lo, hi = np.quantile(all_he, [0.005, 0.995])

plt.figure(figsize=(9, 5))
for name, he in he_dict.items():
    plt.hist(he, bins=80, range=(lo, hi), density=True, histtype="step", linewidth=1.8, label=name)
plt.axvline(0.0, linestyle="--", linewidth=1.0)
plt.xlabel("Hedge error")
plt.ylabel("Density")
plt.title("Heston hedge-error distributions")
plt.legend()
plt.tight_layout()
plt.savefig(Path(OUTPUT_DIR) / "heston_delta_vega_hedge_error_histograms.png", dpi=200)
plt.show()

# Validation loss
plt.figure(figsize=(8, 5))
plt.plot(hist_stock["epoch"], hist_stock["val_loss"], label="Stock-only NN")
plt.plot(hist_stock_option["epoch"], hist_stock_option["val_loss"], label="Stock + option NN")
plt.yscale("log")
plt.xlabel("Epoch")
plt.ylabel("Validation MSE")
plt.title("Heston neural hedge validation loss")
plt.legend()
plt.tight_layout()
plt.savefig(Path(OUTPUT_DIR) / "heston_delta_vega_validation_loss.png", dpi=200)
plt.show()


In [ ]:

# ============================================================
# Position diagnostics by moneyness and variance
# ============================================================

@torch.no_grad()
def binned_positions_by_variable(var_np, positions, time_index, n_bins=25):
    delta = positions["delta"][:, time_index].detach().cpu().numpy()
    eta = positions["eta"][:, time_index].detach().cpu().numpy()

    bins = np.linspace(np.quantile(var_np, 0.01), np.quantile(var_np, 0.99), n_bins)
    mids = 0.5 * (bins[:-1] + bins[1:])
    avg_delta, avg_eta = [], []

    for a, b in zip(bins[:-1], bins[1:]):
        mask = (var_np >= a) & (var_np < b)
        if mask.sum() == 0:
            avg_delta.append(np.nan)
            avg_eta.append(np.nan)
        else:
            avg_delta.append(np.mean(delta[mask]))
            avg_eta.append(np.mean(eta[mask]))

    return mids, np.array(avg_delta), np.array(avg_eta)


mid_index = N_STEPS // 2

with torch.no_grad():
    _, pos_stock = hedge_error_stock_only(stock_model, S_test, v_test, C_test, payoff_test, times)
    _, pos_stock_option = hedge_error_stock_option(stock_option_model, S_test, v_test, C_test, payoff_test, times)

log_m = torch.log(S_test[:, mid_index] / K_TARGET).detach().cpu().numpy()
variance_mid = v_test[:, mid_index].detach().cpu().numpy()

m_mids, stock_delta_m, _ = binned_positions_by_variable(log_m, pos_stock, mid_index)
_, so_delta_m, so_eta_m = binned_positions_by_variable(log_m, pos_stock_option, mid_index)
_, proxy_delta_m, _ = binned_positions_by_variable(log_m, {"delta": proxy_delta, "eta": proxy_eta_zero}, mid_index)
_, dv_delta_m, dv_eta_m = binned_positions_by_variable(log_m, {"delta": dv_delta, "eta": dv_eta}, mid_index)

# Stock position by moneyness
plt.figure(figsize=(9, 5))
plt.plot(m_mids, proxy_delta_m, marker="o", label="BS proxy stock delta")
plt.plot(m_mids, dv_delta_m, marker="o", label="BS proxy delta-vega: stock")
plt.plot(m_mids, stock_delta_m, marker="o", label="Stock-only NN")
plt.plot(m_mids, so_delta_m, marker="o", label="Stock + option NN: stock")
plt.xlabel("Log-moneyness log(S/K)")
plt.ylabel("Average stock position")
plt.title("Average stock position by moneyness")
plt.legend()
plt.tight_layout()
plt.savefig(Path(OUTPUT_DIR) / "heston_stock_position_by_moneyness.png", dpi=200)
plt.show()

# Option position by moneyness
plt.figure(figsize=(9, 5))
plt.plot(m_mids, dv_eta_m, marker="o", label="BS proxy delta-vega: option")
plt.plot(m_mids, so_eta_m, marker="o", label="Stock + option NN: option")
plt.axhline(0.0, linestyle="--", linewidth=1.0)
plt.xlabel("Log-moneyness log(S/K)")
plt.ylabel("Average option position")
plt.title("Average liquid-option position by moneyness")
plt.legend()
plt.tight_layout()
plt.savefig(Path(OUTPUT_DIR) / "heston_option_position_by_moneyness.png", dpi=200)
plt.show()

# Option position by variance
v_mids, _, so_eta_v = binned_positions_by_variable(variance_mid, pos_stock_option, mid_index)
_, _, dv_eta_v = binned_positions_by_variable(variance_mid, {"delta": dv_delta, "eta": dv_eta}, mid_index)

plt.figure(figsize=(9, 5))
plt.plot(v_mids, dv_eta_v, marker="o", label="BS proxy delta-vega: option")
plt.plot(v_mids, so_eta_v, marker="o", label="Stock + option NN: option")
plt.axhline(0.0, linestyle="--", linewidth=1.0)
plt.xlabel("Variance v")
plt.ylabel("Average option position")
plt.title("Average liquid-option position by variance")
plt.legend()
plt.tight_layout()
plt.savefig(Path(OUTPUT_DIR) / "heston_option_position_by_variance.png", dpi=200)
plt.show()

diagnostic_df = pd.DataFrame({
    "log_moneyness_mid": m_mids,
    "proxy_stock_delta": proxy_delta_m,
    "proxy_dv_delta": dv_delta_m,
    "proxy_dv_eta": dv_eta_m,
    "stock_nn_delta": stock_delta_m,
    "stock_option_nn_delta": so_delta_m,
    "stock_option_nn_eta": so_eta_m,
})
diagnostic_df.to_csv(Path(OUTPUT_DIR) / "heston_positions_by_moneyness.csv", index=False)
display(diagnostic_df.head())


In [ ]:

# ============================================================
# LaTeX table helper
# ============================================================

latex_cols = [
    "Strategy",
    "Premium",
    "RMSE",
    "Loss CVaR95",
    "Loss CVaR99",
    "Mean abs delta",
    "Mean abs eta",
    "Mean stock turnover",
    "Mean option turnover",
]

latex_df = results_df[latex_cols].copy()
print(latex_df.to_latex(index=False, float_format=lambda x: f"{x:.6f}", escape=False))



## Report wording guidance

Use cautious wording.

Good wording:

> We extend the hedging problem to a stochastic-volatility setting by simulating Heston dynamics and allowing the neural strategy to trade both the underlying stock and a liquid option proxy. In contrast to the Black--Scholes setting, the second option is not merely a redundant instrument: it provides exposure to volatility risk and therefore plays the role of a vega-hedging asset.

Important caveat:

> The liquid option price is approximated using a Black--Scholes proxy with instantaneous volatility \(\sqrt{v_t}\), rather than exact Heston Fourier pricing. The experiment should therefore be interpreted as a practical stock-and-option hedging proxy under stochastic volatility, not as an exact Heston calibration exercise.

Avoid saying:

> The neural hedge proves that the option is exactly vega hedging.

Safer:

> The learned option position is consistent with the use of a liquid option to manage volatility exposure under stochastic volatility.



## Report output manifest

After running the notebook, the following files should be available for the report.

Recommended main-table files:

- `heston_delta_vega_results.csv`
- `heston_delta_vega_outputs/heston_delta_vega_results.csv`
- `heston_delta_vega_outputs/heston_delta_vega_improvement_summary.csv`
- `heston_delta_vega_outputs/heston_parameter_table.csv`

Recommended main-report figures:

- `heston_delta_vega_outputs/heston_sample_stock_paths.png`
- `heston_delta_vega_outputs/heston_sample_variance_paths.png`
- `heston_delta_vega_outputs/heston_delta_vega_rmse_bar.png`
- `heston_delta_vega_outputs/heston_delta_vega_cvar95_bar.png`
- `heston_delta_vega_outputs/heston_option_position_by_variance.png`

Optional supporting/appendix figures:

- `heston_delta_vega_outputs/heston_delta_vega_hedge_error_histograms.png`
- `heston_delta_vega_outputs/heston_stock_position_by_moneyness.png`
- `heston_delta_vega_outputs/heston_option_position_by_moneyness.png`
- `heston_delta_vega_outputs/heston_delta_vega_validation_loss.png`


In [ ]:

# ============================================================
# Report output manifest
# ============================================================

manifest = pd.DataFrame([
    {"File": "heston_delta_vega_results.csv", "Use": "Main results table"},
    {"File": str(Path(OUTPUT_DIR) / "heston_delta_vega_results.csv"), "Use": "Main results table copy"},
    {"File": str(Path(OUTPUT_DIR) / "heston_delta_vega_improvement_summary.csv"), "Use": "Percentage improvement summary"},
    {"File": str(Path(OUTPUT_DIR) / "heston_parameter_table.csv"), "Use": "Heston parameter table"},
    {"File": str(Path(OUTPUT_DIR) / "heston_sample_stock_paths.png"), "Use": "Show Heston stock dynamics"},
    {"File": str(Path(OUTPUT_DIR) / "heston_sample_variance_paths.png"), "Use": "Show stochastic variance dynamics"},
    {"File": str(Path(OUTPUT_DIR) / "heston_delta_vega_rmse_bar.png"), "Use": "Main RMSE comparison"},
    {"File": str(Path(OUTPUT_DIR) / "heston_delta_vega_cvar95_bar.png"), "Use": "Main seller-tail-risk comparison"},
    {"File": str(Path(OUTPUT_DIR) / "heston_option_position_by_variance.png"), "Use": "Interpretation of liquid-option/vega exposure"},
    {"File": str(Path(OUTPUT_DIR) / "heston_delta_vega_hedge_error_histograms.png"), "Use": "Optional hedge-error distribution"},
    {"File": str(Path(OUTPUT_DIR) / "heston_stock_position_by_moneyness.png"), "Use": "Optional stock-position diagnostic"},
    {"File": str(Path(OUTPUT_DIR) / "heston_option_position_by_moneyness.png"), "Use": "Optional option-position diagnostic"},
    {"File": str(Path(OUTPUT_DIR) / "heston_delta_vega_validation_loss.png"), "Use": "Optional training diagnostic"},
])

manifest.to_csv(Path(OUTPUT_DIR) / "heston_report_output_manifest.csv", index=False)
display(manifest)



# Technical revision cells

These cells respond to the independent review of the Heston stock-and-option extension.

They add:

1. proxy option drift diagnostics;
2. a stock-only tanh-output NN to remove the sigmoid-vs-tanh output confound;
3. symmetric mean-optimal evaluation premiums for all strategies;
4. raw/as-trained premium diagnostics for mean hedge-error bias;
5. \(\eta\)-saturation diagnostics;
6. revised report plots and CSV files.

These cells assume the notebook above has been run first, so that the original stock-only model, stock+option model, simulated paths, and helper functions already exist.


In [ ]:

# ============================================================
# REVISION 1: Proxy option drift diagnostics
# ============================================================

def proxy_drift_diagnostics(C, label):
    dC = C[:, 1:] - C[:, :-1]
    flat = dC.detach().cpu().numpy().reshape(-1)
    total = (C[:, -1] - C[:, 0]).detach().cpu().numpy()

    return {
        "Dataset": label,
        "Mean dC per step": float(flat.mean()),
        "Std dC per step": float(flat.std(ddof=1)),
        "SE mean dC per step": float(flat.std(ddof=1) / np.sqrt(flat.size)),
        "Mean total dC over target horizon": float(total.mean()),
        "Std total dC over target horizon": float(total.std(ddof=1)),
        "SE mean total dC": float(total.std(ddof=1) / np.sqrt(total.size)),
    }

proxy_drift_df = pd.DataFrame([
    proxy_drift_diagnostics(C_train, "train"),
    proxy_drift_diagnostics(C_val, "validation"),
    proxy_drift_diagnostics(C_test, "test"),
])

proxy_drift_df.to_csv(Path(OUTPUT_DIR) / "heston_proxy_option_drift_diagnostics.csv", index=False)
display(proxy_drift_df)

print(
    "If the proxy option has material non-zero drift under the Heston simulation measure, "
    "stock+option NN improvements must be interpreted with this caveat."
)


In [ ]:

# ============================================================
# REVISION 2: Stock-only tanh NN to remove output-parametrization confound
# ============================================================

class StockOnlyHestonTanhNN(nn.Module):
    """
    Stock-only Heston neural hedge with tanh-scaled unconstrained stock position.
    This is included to test whether any stock+option improvement is due to the
    extra traded option or merely due to the tanh output parametrization.
    """
    def __init__(self, hidden_width=64, hidden_depth=3, delta_scale=DELTA_SCALE):
        super().__init__()
        self.delta_scale = delta_scale

        layers = []
        in_dim = 3
        for _ in range(hidden_depth):
            layers.append(nn.Linear(in_dim, hidden_width))
            layers.append(nn.Tanh())
            in_dim = hidden_width
        layers.append(nn.Linear(hidden_width, 1))
        self.net = nn.Sequential(*layers)
        self.premium = nn.Parameter(torch.tensor(target_premium_mc, dtype=torch.float32))

    def forward(self, x):
        B, N, D = x.shape
        raw = self.net(x.reshape(B * N, D)).reshape(B, N)
        return self.delta_scale * torch.tanh(raw)

print("Training additional stock-only tanh NN...")
stock_tanh_model = StockOnlyHestonTanhNN(HIDDEN_WIDTH, HIDDEN_DEPTH, DELTA_SCALE).to(device)

# The original train_model uses kind='stock' for stock-only hedge_error_stock_only.
stock_tanh_model, hist_stock_tanh = train_model(
    stock_tanh_model,
    "stock",
    S_train, v_train, C_train, payoff_train,
    S_val, v_val, C_val, payoff_val,
    times,
)

hist_stock_tanh.to_csv(Path(OUTPUT_DIR) / "stock_only_tanh_training_history.csv", index=False)


In [ ]:

# ============================================================
# REVISION 3: Symmetric fair-premium and raw-premium evaluation
# ============================================================

@torch.no_grad()
def position_turnover_revision(pos):
    prev = torch.cat([torch.zeros(pos.shape[0], 1, device=pos.device), pos[:, :-1]], dim=1)
    return torch.sum(torch.abs(pos - prev), dim=1)

@torch.no_grad()
def gains_from_positions_revision(delta, eta, S, C_h):
    dS = S[:, 1:] - S[:, :-1]
    dC = C_h[:, 1:] - C_h[:, :-1]
    return torch.sum(delta * dS + eta * dC, dim=1)

@torch.no_grad()
def gains_stock_only_model_revision(model, S, v, C_h, payoff, times):
    he, positions = hedge_error_stock_only(model, S, v, C_h, payoff, times)
    # hedge_error_stock_only returns he = premium + gains - payoff.
    gains = he - model.premium + payoff
    return gains, positions

@torch.no_grad()
def gains_stock_option_model_revision(model, S, v, C_h, payoff, times):
    he, positions = hedge_error_stock_option(model, S, v, C_h, payoff, times)
    gains = he - model.premium + payoff
    return gains, positions

def summarize_from_gains_revision(name, gains, positions, payoff, premium, premium_mode):
    """
    premium_mode='fair': use mean-optimal evaluation premium E[payoff - gains].
    premium_mode='raw': use supplied premium, e.g. MC premium or learned premium.
    """
    if premium_mode == "fair":
        eval_premium = float(torch.mean(payoff - gains).detach().cpu())
    elif premium_mode == "raw":
        eval_premium = float(premium)
    else:
        raise ValueError("premium_mode must be 'fair' or 'raw'")

    he = eval_premium + gains - payoff
    he_np = he.detach().cpu().numpy()
    loss = -he_np

    delta = positions["delta"]
    eta = positions["eta"]
    stock_turn = position_turnover_revision(delta).detach().cpu().numpy()
    option_turn = position_turnover_revision(eta).detach().cpu().numpy()

    def cvar_upper(x, alpha):
        q = np.quantile(x, alpha)
        return x[x >= q].mean()

    return {
        "Strategy": name,
        "Premium mode": premium_mode,
        "Premium": eval_premium,
        "RMSE": float(np.sqrt(np.mean(he_np**2))),
        "Mean HE": float(np.mean(he_np)),
        "Std HE": float(np.std(he_np)),
        "HE q01": float(np.quantile(he_np, 0.01)),
        "HE q05": float(np.quantile(he_np, 0.05)),
        "Loss VaR95": float(np.quantile(loss, 0.95)),
        "Loss CVaR95": float(cvar_upper(loss, 0.95)),
        "Loss VaR99": float(np.quantile(loss, 0.99)),
        "Loss CVaR99": float(cvar_upper(loss, 0.99)),
        "Mean abs delta": float(torch.mean(torch.abs(delta)).detach().cpu()),
        "Mean abs eta": float(torch.mean(torch.abs(eta)).detach().cpu()),
        "Mean stock turnover": float(np.mean(stock_turn)),
        "Mean option turnover": float(np.mean(option_turn)),
        "Median stock turnover": float(np.median(stock_turn)),
        "Median option turnover": float(np.median(option_turn)),
    }

with torch.no_grad():
    zero = torch.zeros(N_TEST, N_STEPS, device=device)

    strategy_objects = []

    # No hedge
    gains_no = torch.zeros(N_TEST, device=device)
    strategy_objects.append(("No hedge", gains_no, {"delta": zero, "eta": zero}, target_premium_mc))

    # BS proxy stock delta
    proxy_delta, proxy_eta_zero = proxy_stock_delta_positions(S_test, v_test, times)
    gains_proxy_delta = gains_from_positions_revision(proxy_delta, proxy_eta_zero, S_test, C_test)
    strategy_objects.append(("BS proxy stock delta", gains_proxy_delta, {"delta": proxy_delta, "eta": proxy_eta_zero}, target_premium_mc))

    # BS proxy delta-vega
    dv_delta, dv_eta = proxy_delta_vega_positions(S_test, v_test, times)
    gains_proxy_dv = gains_from_positions_revision(dv_delta, dv_eta, S_test, C_test)
    strategy_objects.append((
        f"BS proxy delta-vega hedge (eta clipped {ETA_CLIP:g})",
        gains_proxy_dv,
        {"delta": dv_delta, "eta": dv_eta},
        target_premium_mc,
    ))

    # Original stock-only sigmoid NN
    gains_stock_sigmoid, pos_stock_sigmoid = gains_stock_only_model_revision(
        stock_model, S_test, v_test, C_test, payoff_test, times
    )
    strategy_objects.append((
        "Stock-only NN sigmoid delta",
        gains_stock_sigmoid,
        pos_stock_sigmoid,
        float(stock_model.premium.detach().cpu()),
    ))

    # New stock-only tanh NN
    gains_stock_tanh, pos_stock_tanh = gains_stock_only_model_revision(
        stock_tanh_model, S_test, v_test, C_test, payoff_test, times
    )
    strategy_objects.append((
        "Stock-only NN tanh delta",
        gains_stock_tanh,
        pos_stock_tanh,
        float(stock_tanh_model.premium.detach().cpu()),
    ))

    # Original stock + option NN
    gains_stock_option, pos_stock_option = gains_stock_option_model_revision(
        stock_option_model, S_test, v_test, C_test, payoff_test, times
    )
    strategy_objects.append((
        "Stock + option NN",
        gains_stock_option,
        pos_stock_option,
        float(stock_option_model.premium.detach().cpu()),
    ))

results_fair_df = pd.DataFrame([
    summarize_from_gains_revision(name, gains, pos, payoff_test, premium, premium_mode="fair")
    for name, gains, pos, premium in strategy_objects
])

results_raw_df = pd.DataFrame([
    summarize_from_gains_revision(name, gains, pos, payoff_test, premium, premium_mode="raw")
    for name, gains, pos, premium in strategy_objects
])

results_fair_df.to_csv(Path(OUTPUT_DIR) / "heston_revised_results_fair_premium.csv", index=False)
results_raw_df.to_csv(Path(OUTPUT_DIR) / "heston_revised_results_raw_premium.csv", index=False)

display(results_fair_df)
display(results_raw_df)


In [ ]:

# ============================================================
# REVISION 4: Improvement summary, confound check, and eta saturation
# ============================================================

def pct_improvement(old, new):
    return 100.0 * (old - new) / old

fair = results_fair_df.set_index("Strategy")

stock_sigmoid = fair.loc["Stock-only NN sigmoid delta"]
stock_tanh = fair.loc["Stock-only NN tanh delta"]
stock_option = fair.loc["Stock + option NN"]
proxy_stock = fair.loc["BS proxy stock delta"]
proxy_dv = fair[fair.index.str.startswith("BS proxy delta-vega")].iloc[0]

improvement_summary = pd.DataFrame([
    {
        "Comparison": "Stock + option NN vs stock-only sigmoid NN",
        "RMSE improvement (%)": pct_improvement(stock_sigmoid["RMSE"], stock_option["RMSE"]),
        "Loss CVaR95 improvement (%)": pct_improvement(stock_sigmoid["Loss CVaR95"], stock_option["Loss CVaR95"]),
    },
    {
        "Comparison": "Stock + option NN vs stock-only tanh NN",
        "RMSE improvement (%)": pct_improvement(stock_tanh["RMSE"], stock_option["RMSE"]),
        "Loss CVaR95 improvement (%)": pct_improvement(stock_tanh["Loss CVaR95"], stock_option["Loss CVaR95"]),
    },
    {
        "Comparison": "Stock-only tanh NN vs stock-only sigmoid NN",
        "RMSE improvement (%)": pct_improvement(stock_sigmoid["RMSE"], stock_tanh["RMSE"]),
        "Loss CVaR95 improvement (%)": pct_improvement(stock_sigmoid["Loss CVaR95"], stock_tanh["Loss CVaR95"]),
    },
    {
        "Comparison": "Stock + option NN vs BS proxy stock delta",
        "RMSE improvement (%)": pct_improvement(proxy_stock["RMSE"], stock_option["RMSE"]),
        "Loss CVaR95 improvement (%)": pct_improvement(proxy_stock["Loss CVaR95"], stock_option["Loss CVaR95"]),
    },
    {
        "Comparison": "Stock + option NN vs BS proxy delta-vega",
        "RMSE improvement (%)": pct_improvement(proxy_dv["RMSE"], stock_option["RMSE"]),
        "Loss CVaR95 improvement (%)": pct_improvement(proxy_dv["Loss CVaR95"], stock_option["Loss CVaR95"]),
    },
])

improvement_summary.to_csv(Path(OUTPUT_DIR) / "heston_revised_improvement_summary.csv", index=False)
display(improvement_summary)

def eta_saturation_diagnostics(name, eta, scale):
    eta_abs = torch.abs(eta).detach().cpu().numpy()
    return {
        "Strategy": name,
        "Scale": scale,
        "Max abs eta": float(np.max(eta_abs)),
        "Mean abs eta": float(np.mean(eta_abs)),
        "Frac |eta| > 0.95*scale": float(np.mean(eta_abs > 0.95 * scale)),
        "Frac |eta| > 0.99*scale": float(np.mean(eta_abs > 0.99 * scale)),
    }

eta_diag = pd.DataFrame([
    eta_saturation_diagnostics("BS proxy delta-vega hedge", dv_eta, ETA_CLIP),
    eta_saturation_diagnostics("Stock + option NN", pos_stock_option["eta"], ETA_SCALE),
])

eta_diag.to_csv(Path(OUTPUT_DIR) / "heston_eta_saturation_diagnostics.csv", index=False)
display(eta_diag)


In [ ]:

# ============================================================
# REVISION 5: Revised report figures
# ============================================================

# Fair-premium RMSE bar plot
plot_df = results_fair_df.sort_values("RMSE")
plt.figure(figsize=(9, 5))
plt.bar(plot_df["Strategy"], plot_df["RMSE"])
plt.xticks(rotation=30, ha="right")
plt.ylabel("RMSE")
plt.title("Heston stock-and-option hedging: fair-premium RMSE")
plt.tight_layout()
plt.savefig(Path(OUTPUT_DIR) / "heston_revised_rmse_bar.png", dpi=200)
plt.show()

# Fair-premium Loss CVaR95
plot_df = results_fair_df.sort_values("Loss CVaR95")
plt.figure(figsize=(9, 5))
plt.bar(plot_df["Strategy"], plot_df["Loss CVaR95"])
plt.xticks(rotation=30, ha="right")
plt.ylabel("Loss CVaR95")
plt.title("Heston stock-and-option hedging: seller tail risk")
plt.tight_layout()
plt.savefig(Path(OUTPUT_DIR) / "heston_revised_cvar95_bar.png", dpi=200)
plt.show()

# Raw Mean HE diagnostic
plt.figure(figsize=(9, 5))
plt.bar(results_raw_df["Strategy"], results_raw_df["Mean HE"])
plt.axhline(0.0, linestyle="--", linewidth=1.0)
plt.xticks(rotation=30, ha="right")
plt.ylabel("Mean hedge error")
plt.title("Raw/as-trained premium mean hedge-error diagnostic")
plt.tight_layout()
plt.savefig(Path(OUTPUT_DIR) / "heston_revised_raw_mean_he_bar.png", dpi=200)
plt.show()

# Fair-premium hedge-error histograms
he_dict = {}
with torch.no_grad():
    for name, gains, pos, premium in strategy_objects:
        fair_premium = torch.mean(payoff_test - gains)
        he = fair_premium + gains - payoff_test
        he_dict[name] = he.detach().cpu().numpy()

key_names = [
    "BS proxy stock delta",
    f"BS proxy delta-vega hedge (eta clipped {ETA_CLIP:g})",
    "Stock-only NN sigmoid delta",
    "Stock-only NN tanh delta",
    "Stock + option NN",
]

all_he = np.concatenate([he_dict[k] for k in key_names])
lo, hi = np.quantile(all_he, [0.005, 0.995])

plt.figure(figsize=(9, 5))
for name in key_names:
    plt.hist(he_dict[name], bins=80, range=(lo, hi), density=True, histtype="step", linewidth=1.8, label=name)
plt.axvline(0.0, linestyle="--", linewidth=1.0)
plt.xlabel("Hedge error")
plt.ylabel("Density")
plt.title("Fair-premium hedge-error distributions")
plt.legend()
plt.tight_layout()
plt.savefig(Path(OUTPUT_DIR) / "heston_revised_hedge_error_histograms.png", dpi=200)
plt.show()


In [ ]:

# ============================================================
# REVISION 6: Position diagnostics for vega-interpretation check
# ============================================================

@torch.no_grad()
def binned_positions_by_variable_revision(var_np, positions, time_index, n_bins=25):
    delta = positions["delta"][:, time_index].detach().cpu().numpy()
    eta = positions["eta"][:, time_index].detach().cpu().numpy()

    bins = np.linspace(np.quantile(var_np, 0.01), np.quantile(var_np, 0.99), n_bins)
    mids = 0.5 * (bins[:-1] + bins[1:])
    avg_delta, avg_eta = [], []

    for a, b in zip(bins[:-1], bins[1:]):
        mask = (var_np >= a) & (var_np < b)
        if mask.sum() == 0:
            avg_delta.append(np.nan)
            avg_eta.append(np.nan)
        else:
            avg_delta.append(np.mean(delta[mask]))
            avg_eta.append(np.mean(eta[mask]))

    return mids, np.array(avg_delta), np.array(avg_eta)

mid_index = N_STEPS // 2
log_m = torch.log(S_test[:, mid_index] / K_TARGET).detach().cpu().numpy()
variance_mid = v_test[:, mid_index].detach().cpu().numpy()

m_mids, sig_delta_m, _ = binned_positions_by_variable_revision(log_m, pos_stock_sigmoid, mid_index)
_, tanh_delta_m, _ = binned_positions_by_variable_revision(log_m, pos_stock_tanh, mid_index)
_, so_delta_m, so_eta_m = binned_positions_by_variable_revision(log_m, pos_stock_option, mid_index)
_, proxy_delta_m, _ = binned_positions_by_variable_revision(log_m, {"delta": proxy_delta, "eta": proxy_eta_zero}, mid_index)
_, dv_delta_m, dv_eta_m = binned_positions_by_variable_revision(log_m, {"delta": dv_delta, "eta": dv_eta}, mid_index)

plt.figure(figsize=(9, 5))
plt.plot(m_mids, proxy_delta_m, marker="o", label="BS proxy stock delta")
plt.plot(m_mids, dv_delta_m, marker="o", label="BS proxy delta-vega: stock")
plt.plot(m_mids, sig_delta_m, marker="o", label="Stock-only sigmoid NN")
plt.plot(m_mids, tanh_delta_m, marker="o", label="Stock-only tanh NN")
plt.plot(m_mids, so_delta_m, marker="o", label="Stock + option NN: stock")
plt.xlabel("Log-moneyness log(S/K)")
plt.ylabel("Average stock position")
plt.title("Average stock position by moneyness")
plt.legend()
plt.tight_layout()
plt.savefig(Path(OUTPUT_DIR) / "heston_revised_stock_position_by_moneyness.png", dpi=200)
plt.show()

plt.figure(figsize=(9, 5))
plt.plot(m_mids, dv_eta_m, marker="o", label="BS proxy delta-vega: option")
plt.plot(m_mids, so_eta_m, marker="o", label="Stock + option NN: option")
plt.axhline(0.0, linestyle="--", linewidth=1.0)
plt.xlabel("Log-moneyness log(S/K)")
plt.ylabel("Average option position")
plt.title("Average liquid-option position by moneyness")
plt.legend()
plt.tight_layout()
plt.savefig(Path(OUTPUT_DIR) / "heston_revised_option_position_by_moneyness.png", dpi=200)
plt.show()

v_mids, _, so_eta_v = binned_positions_by_variable_revision(variance_mid, pos_stock_option, mid_index)
_, _, dv_eta_v = binned_positions_by_variable_revision(variance_mid, {"delta": dv_delta, "eta": dv_eta}, mid_index)

plt.figure(figsize=(9, 5))
plt.plot(v_mids, dv_eta_v, marker="o", label="BS proxy delta-vega: option")
plt.plot(v_mids, so_eta_v, marker="o", label="Stock + option NN: option")
plt.axhline(0.0, linestyle="--", linewidth=1.0)
plt.xlabel("Variance v")
plt.ylabel("Average option position")
plt.title("Average liquid-option position by variance")
plt.legend()
plt.tight_layout()
plt.savefig(Path(OUTPUT_DIR) / "heston_revised_option_position_by_variance.png", dpi=200)
plt.show()

variance_position_df = pd.DataFrame({
    "variance_mid": v_mids,
    "proxy_delta_vega_eta": dv_eta_v,
    "stock_option_nn_eta": so_eta_v,
})
variance_position_df.to_csv(Path(OUTPUT_DIR) / "heston_revised_option_position_by_variance.csv", index=False)
display(variance_position_df.head())


In [ ]:

# ============================================================
# REVISION 7: LaTeX helpers and output manifest
# ============================================================

latex_cols = [
    "Strategy",
    "Premium",
    "RMSE",
    "Mean HE",
    "Loss CVaR95",
    "Loss CVaR99",
    "Mean abs delta",
    "Mean abs eta",
    "Mean stock turnover",
    "Mean option turnover",
]

print("Fair-premium main table:")
print(results_fair_df[latex_cols].to_latex(index=False, float_format=lambda x: f"{x:.6f}", escape=False))

print("\nRaw/as-trained premium diagnostic table:")
print(results_raw_df[["Strategy", "Premium", "RMSE", "Mean HE", "Loss CVaR95"]].to_latex(
    index=False,
    float_format=lambda x: f"{x:.6f}",
    escape=False,
))

manifest = pd.DataFrame([
    {"File": str(Path(OUTPUT_DIR) / "heston_revised_results_fair_premium.csv"), "Use": "Main fair-premium results table"},
    {"File": str(Path(OUTPUT_DIR) / "heston_revised_results_raw_premium.csv"), "Use": "Raw/as-trained premium diagnostics"},
    {"File": str(Path(OUTPUT_DIR) / "heston_proxy_option_drift_diagnostics.csv"), "Use": "Proxy option drift diagnostic"},
    {"File": str(Path(OUTPUT_DIR) / "heston_revised_improvement_summary.csv"), "Use": "Improvement and parametrization-confound summary"},
    {"File": str(Path(OUTPUT_DIR) / "heston_eta_saturation_diagnostics.csv"), "Use": "Option-position rail saturation diagnostic"},
    {"File": str(Path(OUTPUT_DIR) / "heston_revised_rmse_bar.png"), "Use": "Main RMSE figure"},
    {"File": str(Path(OUTPUT_DIR) / "heston_revised_cvar95_bar.png"), "Use": "Main Loss CVaR95 figure"},
    {"File": str(Path(OUTPUT_DIR) / "heston_revised_raw_mean_he_bar.png"), "Use": "Raw Mean HE diagnostic figure"},
    {"File": str(Path(OUTPUT_DIR) / "heston_revised_option_position_by_variance.png"), "Use": "Vega interpretation diagnostic"},
    {"File": str(Path(OUTPUT_DIR) / "heston_revised_hedge_error_histograms.png"), "Use": "Optional hedge-error histogram"},
    {"File": str(Path(OUTPUT_DIR) / "heston_revised_stock_position_by_moneyness.png"), "Use": "Optional stock-position diagnostic"},
    {"File": str(Path(OUTPUT_DIR) / "heston_revised_option_position_by_moneyness.png"), "Use": "Optional option-position diagnostic"},
])
manifest.to_csv(Path(OUTPUT_DIR) / "heston_revised_report_output_manifest.csv", index=False)
display(manifest)



## Optional multi-seed headline robustness check

Run this only if the headline stock+option-vs-stock-only-tanh gap is small enough that it could plausibly sit inside Monte Carlo/training noise.

This is intentionally disabled by default because it retrains the headline models multiple times.


In [ ]:

# ============================================================
# OPTIONAL: Multi-seed headline robustness check
# ============================================================

RUN_MULTI_SEED_HEADLINE = False
MULTI_SEEDS = [111, 222, 333]

def train_model_with_seed(model, kind, S_tr, v_tr, C_tr, y_tr, S_va, v_va, C_va, y_va, times_grid, seed, max_epochs=None):
    """
    Local training helper for the optional multi-seed check.
    Uses the existing hedge_error_stock_only / hedge_error_stock_option functions.
    """
    if max_epochs is None:
        max_epochs = MAX_EPOCHS

    torch.manual_seed(seed)
    model = model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)

    best_val = float("inf")
    best_state = None
    bad_epochs = 0
    n = S_tr.shape[0]

    for epoch in range(1, max_epochs + 1):
        model.train()
        perm = torch.randperm(n, device=device)

        for start in range(0, n, BATCH_SIZE):
            idx = perm[start:start+BATCH_SIZE]
            optimizer.zero_grad()

            if kind == "stock":
                he, _ = hedge_error_stock_only(model, S_tr[idx], v_tr[idx], C_tr[idx], y_tr[idx], times_grid)
            else:
                he, _ = hedge_error_stock_option(model, S_tr[idx], v_tr[idx], C_tr[idx], y_tr[idx], times_grid)

            loss = torch.mean(he ** 2)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
            optimizer.step()

        model.eval()
        with torch.no_grad():
            if kind == "stock":
                he_val, _ = hedge_error_stock_only(model, S_va, v_va, C_va, y_va, times_grid)
            else:
                he_val, _ = hedge_error_stock_option(model, S_va, v_va, C_va, y_va, times_grid)

            val_loss = float(torch.mean(he_val ** 2).detach().cpu())

        if val_loss < best_val - 1e-10:
            best_val = val_loss
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            bad_epochs = 0
        else:
            bad_epochs += 1

        if bad_epochs >= PATIENCE:
            break

    if best_state is not None:
        model.load_state_dict(best_state)

    return model

def summarize_seed_result(strategy_name, gains, positions, payoff):
    """
    Fair-premium summary for one seed.
    """
    fair_premium = torch.mean(payoff - gains)
    he = fair_premium + gains - payoff
    he_np = he.detach().cpu().numpy()
    loss = -he_np

    def cvar_upper(x, alpha):
        q = np.quantile(x, alpha)
        return x[x >= q].mean()

    return {
        "Strategy": strategy_name,
        "RMSE": float(np.sqrt(np.mean(he_np**2))),
        "Loss CVaR95": float(cvar_upper(loss, 0.95)),
        "Loss CVaR99": float(cvar_upper(loss, 0.99)),
    }

if RUN_MULTI_SEED_HEADLINE:
    multi_seed_rows = []

    for base_seed in MULTI_SEEDS:
        print(f"Running headline multi-seed check for seed {base_seed}")

        # Simulate fresh train/val/test sets for the seed.
        S_tr, v_tr, C_tr, t_grid = simulate_heston_paths(N_TRAIN, base_seed + 1)
        S_va, v_va, C_va, _ = simulate_heston_paths(N_VAL, base_seed + 2)
        S_te, v_te, C_te, _ = simulate_heston_paths(N_TEST, base_seed + 3)

        y_tr = torch.clamp(S_tr[:, -1] - K_TARGET, min=0.0)
        y_va = torch.clamp(S_va[:, -1] - K_TARGET, min=0.0)
        y_te = torch.clamp(S_te[:, -1] - K_TARGET, min=0.0)

        # Update global premium initializer used by the model constructors.
        target_premium_mc = float(torch.mean(y_tr).detach().cpu())

        # Train headline models.
        m_tanh = StockOnlyHestonTanhNN(HIDDEN_WIDTH, HIDDEN_DEPTH, DELTA_SCALE)
        m_so = StockOptionHestonNN(HIDDEN_WIDTH, HIDDEN_DEPTH, DELTA_SCALE, ETA_SCALE)

        m_tanh = train_model_with_seed(m_tanh, "stock", S_tr, v_tr, C_tr, y_tr, S_va, v_va, C_va, y_va, t_grid, base_seed + 10)
        m_so = train_model_with_seed(m_so, "stock_option", S_tr, v_tr, C_tr, y_tr, S_va, v_va, C_va, y_va, t_grid, base_seed + 20)

        with torch.no_grad():
            gains_tanh, pos_tanh_ms = gains_stock_only_model_revision(m_tanh, S_te, v_te, C_te, y_te, t_grid)
            gains_so_ms, pos_so_ms = gains_stock_option_model_revision(m_so, S_te, v_te, C_te, y_te, t_grid)

        for row in [
            summarize_seed_result("Stock-only NN tanh delta", gains_tanh, pos_tanh_ms, y_te),
            summarize_seed_result("Stock + option NN", gains_so_ms, pos_so_ms, y_te),
        ]:
            row["Seed"] = base_seed
            multi_seed_rows.append(row)

    multi_seed_df = pd.DataFrame(multi_seed_rows)
    multi_seed_summary = (
        multi_seed_df
        .groupby("Strategy")[["RMSE", "Loss CVaR95", "Loss CVaR99"]]
        .agg(["mean", "std", "min", "max"])
    )

    multi_seed_df.to_csv(Path(OUTPUT_DIR) / "heston_revised_multiseed_headline_raw.csv", index=False)
    multi_seed_summary.to_csv(Path(OUTPUT_DIR) / "heston_revised_multiseed_headline_summary.csv")

    display(multi_seed_df)
    display(multi_seed_summary)
else:
    print("RUN_MULTI_SEED_HEADLINE is False. Set it to True if the headline fair-premium gap is small.")



## Corrected reporting guidance after technical review

Use the safer title unless the diagnostics strongly support the vega interpretation:

```latex
\subsection{Stock-and-Option Neural Hedging under Heston Dynamics}
```

The earlier guidance over-trusted raw Mean HE as a proxy-drift detector. That is now corrected.

### Why raw Mean HE is not a primary drift test

For fair-premium evaluation,

\[
\pi^*=\mathbb{E}[\Phi(S_T)-\text{gains}],
\]

so

\[
\mathbb{E}[HE_{\text{fair}}]
=
\pi^*+\mathbb{E}[\text{gains}]-\mathbb{E}[\Phi(S_T)]
=
0
\]

by construction for every strategy. The fair-premium table therefore cannot diagnose drift.

For a premium-learning neural model evaluated at its learned premium,

\[
\mathbb{E}[HE_{\text{raw}}]
=
\pi_{\text{learn}}-\pi^*.
\]

Thus raw Mean HE mostly measures whether the learned premium has converged to the MSE-optimal premium. A near-zero raw Mean HE for the NN does **not** prove that the proxy option drift is harmless.

Raw Mean HE is still useful as a corroborating diagnostic, especially for fixed-premium benchmarks, but it should not be used as the main decision rule for the stock+option NN.

### Correct decision tree

**Step 1: Check whether the proxy option has drift.**

Use:

```text
heston_proxy_option_drift_diagnostics.csv
```

Compare `Mean dC per step` and `Mean total dC over target horizon` with their reported standard errors.

- If the proxy drift is within about two standard errors of zero, the drift artifact is empirically small.
- If the proxy drift is statistically non-zero, then the proxy instrument is not martingale-consistent under the Heston simulation measure.

**Step 2: If drift exists, ask whether it contaminates the result.**

The definitive test is to reprice the hedge option using a martingale-consistent Heston price, for example COS or Carr--Madan FFT, and rerun the stock+option comparison. This notebook does not implement that exact-Heston repricing step.

The suggestive diagnostic already included here is:

```text
heston_revised_option_position_by_variance.png
```

If the learned NN option-position curve is variance/moneyness-contingent and broadly tracks the proxy delta--vega curve, the vega-hedging interpretation is more plausible. If it is poorly aligned or behaves like a flat offset, then the mechanism is not conclusively identified.

**Step 3: Calibrate the claim.**

If proxy drift is negligible, or if a later COS repricing confirms that the performance gap survives, the more specific title

```latex
\subsection{Delta--Vega Neural Hedging under Heston Dynamics}
```

is defensible.

If proxy drift is material and COS repricing has not been run, use the safer title:

```latex
\subsection{Stock-and-Option Neural Hedging under Heston Dynamics}
```

and state:

> The liquid option is priced using a Black--Scholes instantaneous-volatility proxy. Since this proxy need not be a martingale under the Heston simulation measure, the results are interpreted as a practical stock-and-option hedging experiment under stochastic volatility. The proxy-drift diagnostic and option-position-by-variance plot are reported to assess whether the improvement is consistent with volatility-sensitive hedging, but exact separation from proxy-pricing effects would require martingale-consistent Heston option prices.

### Multi-seed reporting rule

Use the single-seed results only if the stock+option-vs-stock-only-tanh gap is clearly large.

If the RMSE/CVaR gap is small, run the optional multi-seed cell below and report mean \(\pm\) standard deviation, or min/max, for the headline strategies. This is especially important for CVaR99, which is noisy on a finite test set.
